# ExplainPlan-Vision
## Phase 3: Explainable Sequential Planning Engine

**Project:** ExplainPlan-Vision — Explainable Neuro-Symbolic Visual Planning Agent  
**Phase:** 3 of 7 — Explainable Planning Engine  
**Depends on:** Phase 1 outputs (best_model.pth, class_mappings.json, config.json)  
**Author:** Muhammad Aqib Javed 
**Date:** 2026

### What this phase builds

Phases 1 and 2 gave us a system that can see and explain. Phase 3 adds the ability to reason and plan. The transition is significant: we move from answering *what is wrong with this plant* to answering *what should be done about it, in what order, and why*.

This is the core research contribution of ExplainPlan-Vision. Most plant disease AI systems stop at classification. A small number add visual explanations. Almost none produce sequenced, explainable treatment plans grounded in the model's own uncertainty and visual evidence. This phase crosses that boundary.

The planning engine is deliberately rule-based and symbolic. This is a principled choice, not a limitation. A deterministic planner produces plans that can be audited, verified, and explained step by step. An LLM-based planner produces fluent text that cannot be reliably traced back to the model's evidence. For a system intended to guide agricultural treatment decisions — where wrong actions have real costs — auditability is a research requirement, not an aesthetic preference.

### Full system pipeline after Phase 3

```
Image
  Phase 1   Vision backbone (EfficientNet-B0)
             {disease, confidence, severity, embedding}
  Phase 2   XAI layer (Grad-CAM, SHAP, LIME)
             {heatmap, attribution, explanation text}
  Phase 3   Planning engine  <— this phase
             {state, action_sequence, rationale, counterfactuals}
  Phase 5   Human-aware adapter (farmer / researcher / agronomist)
```

### Phase 3 modules

| Module | Purpose |
|--------|---------|
| Structured State Generator | Converts model output into symbolic planning state |
| Disease Knowledge Base | Encodes agronomic rules as structured JSON |
| Rule-Based Planner | Generates sequenced action plans from state + knowledge |
| Priority Scorer | Orders actions by urgency, confidence, and spread risk |
| Counterfactual Planner | Generates alternative plans under different assumptions |
| Human-Aware Explainer | Adapts plan language to three user expertise levels |
| Full Pipeline | End-to-end image → plan function used by Phase 5 and the API |

---
## Kaggle setup for Phase 3

Phase 3 needs the Phase 1 model to run inference on new images. It does not need Phase 2's XAI outputs for its core planning logic, but it does call the Phase 2 `explain()` function to enrich plans with visual evidence.

Add the following datasets as inputs to this notebook:

1. `explainplan-phase1-outputs` — contains `best_model.pth`, `class_mappings.json`, `config.json`
2. `plantdisease` — the PlantVillage dataset, for the end-to-end demo

Phase 2 outputs are not required as a separate dataset input. The planning engine operates on the structured prediction dictionary that Phase 1 already produces. The XAI engines are re-instantiated in this notebook using the same Phase 1 model weights.

GPU time estimate: model inference only, no training. Under 20 minutes for the full notebook.

---
## Cell 1 — Install dependencies

In [ ]:
!pip install -q albumentations timm grad-cam shap lime

---
## Cell 2 — Imports

In [ ]:
import os
import json
import copy
import random
import warnings
import textwrap
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple
from enum import Enum

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.amp import autocast
import timm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

warnings.filterwarnings("ignore")
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

---
## Cell 3 — Configuration and Phase 1 artefact loading

In [ ]:
PHASE1_DIR   = "/kaggle/input/explainplan-phase1-outputs"
DATASET_PATH = "/kaggle/input/plantdisease/PlantVillage"

with open(f"{PHASE1_DIR}/outputs/checkpoints/config.json") as f:
    P1_CONFIG = json.load(f)

with open(f"{PHASE1_DIR}/outputs/checkpoints/class_mappings.json") as f:
    mappings = json.load(f)

class_to_idx = mappings["class_to_idx"]
idx_to_class = {int(k): v for k, v in mappings["idx_to_class"].items()}
classes      = mappings["classes"]
NUM_CLASSES  = mappings["num_classes"]

CONFIG = {
    # Inherited from Phase 1
    "model_name"      : P1_CONFIG["model_name"],
    "image_size"      : P1_CONFIG["image_size"],
    "mean"            : P1_CONFIG["mean"],
    "std"             : P1_CONFIG["std"],
    "embedding_dim"   : P1_CONFIG["embedding_dim"],
    "dropout"         : P1_CONFIG["dropout"],
    "device"          : "cuda" if torch.cuda.is_available() else "cpu",
    "checkpoint_path" : f"{PHASE1_DIR}/outputs/checkpoints/best_model.pth",
    "severity_thresholds": {"high": 0.85, "medium": 0.60, "low": 0.0},

    # Phase 3 specific
    "plan_output_dir" : "outputs/planning",
    "seed"            : 42,
}

for subdir in ["reports", "figures", "json"]:
    os.makedirs(f"{CONFIG['plan_output_dir']}/{subdir}", exist_ok=True)

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

print(f"Phase 1 config loaded")
print(f"  Model      : {CONFIG['model_name']}")
print(f"  Classes    : {NUM_CLASSES}")
print(f"  Device     : {CONFIG['device']}")

---
## Cell 4 — Vision model (Phase 1 architecture, frozen weights)

In [ ]:
inference_transform = A.Compose([
    A.Resize(CONFIG["image_size"], CONFIG["image_size"]),
    A.Normalize(mean=CONFIG["mean"], std=CONFIG["std"]),
    ToTensorV2()
])

def load_image_rgb(path):
    img = cv2.imread(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def preprocess_for_model(image_rgb):
    tensor = inference_transform(image=image_rgb)["image"]
    return tensor.unsqueeze(0).to(CONFIG["device"])

def preprocess_for_display(image_rgb):
    resized = cv2.resize(image_rgb, (CONFIG["image_size"], CONFIG["image_size"]))
    return resized.astype(np.float32) / 255.0


class PlantDiseaseModel(nn.Module):
    """Exact copy of Phase 1 architecture. Must not be modified."""
    def __init__(self, model_name, num_classes, embedding_dim=512, dropout=0.3):
        super().__init__()
        self.backbone   = timm.create_model(model_name, pretrained=False, num_classes=0)
        backbone_out    = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(embedding_dim, num_classes)
        )
    def forward(self, x):
        features = self.backbone(x)
        logits   = self.classifier(features)
        return logits, features
    def extract_features(self, x):
        with torch.no_grad():
            return self.backbone(x)


model = PlantDiseaseModel(
    model_name    = CONFIG["model_name"],
    num_classes   = NUM_CLASSES,
    embedding_dim = CONFIG["embedding_dim"],
    dropout       = CONFIG["dropout"]
).to(CONFIG["device"])

model.load_state_dict(
    torch.load(CONFIG["checkpoint_path"],
               map_location=CONFIG["device"],
               weights_only=True)
)
model.eval()
print(f"Phase 1 model loaded and frozen")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## Cell 5 — Base inference function

In [ ]:
def estimate_severity(confidence):
    t = CONFIG["severity_thresholds"]
    if confidence >= t["high"]:
        return "high"
    elif confidence >= t["medium"]:
        return "medium"
    return "low"


def predict(image_rgb):
    """
    Run Phase 1 inference on a uint8 RGB numpy image.
    Returns the structured prediction dictionary.
    """
    model.eval()
    tensor = preprocess_for_model(image_rgb)

    with torch.no_grad():
        with autocast("cuda", enabled=torch.cuda.is_available()):
            logits, features = model(tensor)
        probs = torch.softmax(logits.float(), dim=1)[0]

    top_idx    = probs.argmax().item()
    confidence = probs[top_idx].item()
    label_name = idx_to_class[top_idx]

    if "___" in label_name:
        parts, plant = label_name.split("___", 1), label_name.split("___")[0]
        plant        = parts[0].replace("_", " ").strip()
        disease_type = parts[1].replace("_", " ").strip() if len(parts) > 1 else label_name
        # Fix the split logic
        split_parts  = label_name.split("___", 1)
        plant        = split_parts[0].replace("_", " ").strip()
        disease_type = split_parts[1].replace("_", " ").strip()
    else:
        tokens       = label_name.split("_")
        plant        = tokens[0].strip()
        disease_type = " ".join(tokens[1:]).strip()

    top3_vals, top3_idxs = torch.topk(probs, k=min(3, NUM_CLASSES))
    top3 = [
        {"class": idx_to_class[i.item()], "confidence": round(v.item(), 4)}
        for v, i in zip(top3_vals, top3_idxs)
    ]

    return {
        "disease"          : label_name,
        "plant"            : plant,
        "disease_type"     : disease_type,
        "confidence"       : round(confidence, 4),
        "severity"         : estimate_severity(confidence),
        "top3_predictions" : top3,
        "feature_embedding": features[0].cpu().float().numpy(),
        "is_healthy"       : "healthy" in label_name.lower(),
    }


print("Inference function ready")

---
## Cell 6 — Grad-CAM engine (Phase 2, re-instantiated)

The Grad-CAM engine from Phase 2 is included here so that the planning engine can access the `focus_score` — a measure of how spatially concentrated the model's attention is. A localised high-activation region implies early-stage, contained infection. A distributed activation pattern implies advanced spread. This spatial signal feeds directly into action prioritization.

In [ ]:
class GradCAMWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        logits, _ = self.model(x)
        return logits


class GradCAMEngine:
    def __init__(self, model, device):
        self.wrapped = GradCAMWrapper(model).to(device)
        self.wrapped.eval()
        self.cam = GradCAMPlusPlus(
            model=self.wrapped,
            target_layers=[self.wrapped.model.backbone.conv_head]
        )

    def generate(self, image_rgb, target_class_idx=None):
        tensor    = preprocess_for_model(image_rgb)
        rgb_float = preprocess_for_display(image_rgb)
        targets   = [ClassifierOutputTarget(target_class_idx)] if target_class_idx is not None else None
        heatmap   = self.cam(input_tensor=tensor, targets=targets)[0]
        overlay   = show_cam_on_image(rgb_float, heatmap, use_rgb=True)
        return heatmap, overlay

    def focus_score(self, heatmap):
        """
        Compute the spatial concentration of activations.

        Returns the fraction of pixels in the top-20% activation region
        relative to the total image area. Low values indicate localised
        infection; high values indicate widespread disease spread.
        Values are in [0, 1]; below 0.08 is considered localised.
        """
        threshold = np.percentile(heatmap, 80)
        return float((heatmap >= threshold).mean())


gradcam_engine = GradCAMEngine(model, CONFIG["device"])
print("Grad-CAM engine ready")

---
## Cell 7 — Disease Knowledge Base

The knowledge base is the symbolic component of the neuro-symbolic architecture. It encodes agronomic domain knowledge as structured JSON that the rule-based planner can query deterministically.

Each disease entry contains:
- `pathogen_type` — fungal, bacterial, viral, or pest. This determines the class of treatment.
- `spread_risk` — how quickly the disease propagates without intervention.
- `environmental_triggers` — conditions that worsen or accelerate spread.
- `primary_actions` — the core treatment sequence.
- `secondary_actions` — supportive measures applied alongside primary treatment.
- `monitoring_actions` — post-treatment observation steps.
- `contraindications` — treatments that must not be applied for this disease.
- `urgency_modifier` — adjusts plan urgency independent of model confidence.

This structure was designed so that Phase 4 (neuro-symbolic reasoning) can reason over it using PDDL-style operators without restructuring.

In [ ]:
KNOWLEDGE_BASE = {

    "Bacterial spot": {
        "pathogen_type"        : "bacterial",
        "pathogen_name"        : "Xanthomonas spp.",
        "spread_risk"          : "high",
        "environmental_triggers": ["rain", "high humidity", "warm temperatures"],
        "primary_actions"      : [
            "Apply copper-based bactericide to all affected plants",
            "Remove and destroy severely infected leaves immediately",
            "Avoid overhead irrigation — switch to drip irrigation",
        ],
        "secondary_actions"    : [
            "Disinfect pruning tools between plants with 70% ethanol",
            "Apply preventive copper spray to neighbouring plants",
            "Improve air circulation through selective pruning",
        ],
        "monitoring_actions"   : [
            "Inspect treated plants every 3 days for 2 weeks",
            "Record lesion count per leaf to track progression",
            "Check neighbouring plants for early symptoms",
        ],
        "contraindications"    : ["fungicide-only treatment", "delayed action"],
        "urgency_modifier"     : 1.2,
        "recovery_timeline"    : "2-3 weeks with consistent treatment",
    },

    "Early blight": {
        "pathogen_type"        : "fungal",
        "pathogen_name"        : "Alternaria solani",
        "spread_risk"          : "medium",
        "environmental_triggers": ["high humidity", "warm days", "cool nights"],
        "primary_actions"      : [
            "Apply copper-based or chlorothalonil fungicide",
            "Remove infected lower leaves and dispose away from field",
            "Reduce overhead irrigation and improve drainage",
        ],
        "secondary_actions"    : [
            "Apply mulch around base to reduce soil splash",
            "Space plants to improve airflow and reduce leaf wetness",
            "Apply potassium fertilizer to strengthen plant resistance",
        ],
        "monitoring_actions"   : [
            "Inspect every 5 days for lesion progression",
            "Monitor for secondary fungal infections",
        ],
        "contraindications"    : ["bactericide-only treatment"],
        "urgency_modifier"     : 1.0,
        "recovery_timeline"    : "3-4 weeks with fungicide rotation",
    },

    "Late blight": {
        "pathogen_type"        : "oomycete",
        "pathogen_name"        : "Phytophthora infestans",
        "spread_risk"          : "critical",
        "environmental_triggers": ["cool temperatures", "high humidity", "leaf wetness"],
        "primary_actions"      : [
            "Isolate infected plants from healthy crops immediately",
            "Apply systemic fungicide (mancozeb or metalaxyl) within 24 hours",
            "Remove and destroy all infected plant material — do not compost",
            "Eliminate overhead irrigation completely",
        ],
        "secondary_actions"    : [
            "Apply preventive fungicide to all neighbouring plants",
            "Install physical barriers to limit spore movement",
            "Monitor weather — act before rain events",
        ],
        "monitoring_actions"   : [
            "Inspect every 2 days — late blight can destroy a crop in 7-10 days",
            "Track weather forecasts for cool, wet conditions",
            "Consider full crop removal if spread exceeds 30% of plants",
        ],
        "contraindications"    : ["delayed treatment", "contact-only fungicide", "continued overhead irrigation"],
        "urgency_modifier"     : 1.8,
        "recovery_timeline"    : "Aggressive intervention required within 48 hours",
    },

    "Leaf Mold": {
        "pathogen_type"        : "fungal",
        "pathogen_name"        : "Passalora fulva",
        "spread_risk"          : "medium",
        "environmental_triggers": ["humidity above 85%", "poor ventilation", "greenhouse conditions"],
        "primary_actions"      : [
            "Improve greenhouse ventilation immediately",
            "Apply appropriate fungicide (chlorothalonil or mancozeb)",
            "Remove and destroy heavily infected leaves",
        ],
        "secondary_actions"    : [
            "Reduce relative humidity below 85% using ventilation or dehumidifiers",
            "Avoid wetting foliage during irrigation",
            "Increase plant spacing to improve airflow",
        ],
        "monitoring_actions"   : [
            "Check undersides of leaves daily for olive-green mould growth",
            "Monitor humidity levels with hygrometer",
        ],
        "contraindications"    : ["increasing irrigation", "closing ventilation"],
        "urgency_modifier"     : 1.0,
        "recovery_timeline"    : "2-3 weeks with environmental control",
    },

    "Septoria leaf spot": {
        "pathogen_type"        : "fungal",
        "pathogen_name"        : "Septoria lycopersici",
        "spread_risk"          : "medium",
        "environmental_triggers": ["wet weather", "splashing water", "infected debris"],
        "primary_actions"      : [
            "Remove all infected leaves starting from the lowest",
            "Apply fungicide containing chlorothalonil, mancozeb, or copper",
            "Avoid overhead watering to reduce spore splash",
        ],
        "secondary_actions"    : [
            "Apply thick mulch layer to prevent soil splash",
            "Stake plants to keep foliage off the ground",
            "Rotate fungicide classes to prevent resistance",
        ],
        "monitoring_actions"   : [
            "Inspect every 5-7 days",
            "Track percentage of defoliated leaves as a severity metric",
        ],
        "contraindications"    : ["overhead irrigation continuation"],
        "urgency_modifier"     : 1.0,
        "recovery_timeline"    : "3-5 weeks with consistent fungicide programme",
    },

    "Spider mites Two spotted spider mite": {
        "pathogen_type"        : "pest",
        "pathogen_name"        : "Tetranychus urticae",
        "spread_risk"          : "high",
        "environmental_triggers": ["hot dry conditions", "dusty environments", "drought stress"],
        "primary_actions"      : [
            "Apply miticide or insecticidal soap to all leaf surfaces including undersides",
            "Remove heavily infested leaves",
            "Increase humidity — spider mites thrive in dry conditions",
        ],
        "secondary_actions"    : [
            "Release predatory mites (Phytoseiulus persimilis) as biological control",
            "Spray plants with water to physically dislodge mites",
            "Avoid broad-spectrum pesticides that kill natural predators",
        ],
        "monitoring_actions"   : [
            "Use hand lens to inspect leaf undersides every 3 days",
            "Check for webbing as indicator of severe infestation",
        ],
        "contraindications"    : ["fungicide treatment", "broad-spectrum pesticides killing predators"],
        "urgency_modifier"     : 1.3,
        "recovery_timeline"    : "2-3 weeks with combined chemical and biological control",
    },

    "Target Spot": {
        "pathogen_type"        : "fungal",
        "pathogen_name"        : "Corynespora cassiicola",
        "spread_risk"          : "medium",
        "environmental_triggers": ["warm temperatures", "high humidity", "wet foliage"],
        "primary_actions"      : [
            "Apply chlorothalonil or azoxystrobin fungicide",
            "Remove infected plant material",
            "Improve air circulation within the canopy",
        ],
        "secondary_actions"    : [
            "Avoid excessive nitrogen fertilisation",
            "Maintain consistent irrigation to avoid moisture stress",
        ],
        "monitoring_actions"   : [
            "Inspect every 5-7 days",
            "Monitor for progression to fruit infection",
        ],
        "contraindications"    : ["excessive nitrogen application"],
        "urgency_modifier"     : 0.9,
        "recovery_timeline"    : "3-4 weeks with fungicide programme",
    },

    "Tomato YellowLeaf  Curl Virus": {
        "pathogen_type"        : "viral",
        "pathogen_name"        : "TYLCV (Tomato Yellow Leaf Curl Virus)",
        "spread_risk"          : "critical",
        "environmental_triggers": ["whitefly presence", "warm temperatures", "dry conditions"],
        "primary_actions"      : [
            "Remove and destroy all infected plants — no chemical cure exists",
            "Apply systemic insecticide to control whitefly vector immediately",
            "Install yellow sticky traps to monitor and reduce whitefly population",
            "Install fine mesh screens on greenhouse vents",
        ],
        "secondary_actions"    : [
            "Plant virus-resistant tomato varieties for next season",
            "Establish a whitefly monitoring programme",
            "Remove weeds that serve as whitefly reservoir hosts",
        ],
        "monitoring_actions"   : [
            "Check new growth for curl and yellowing symptoms daily",
            "Count whiteflies per plant weekly using sticky traps",
        ],
        "contraindications"    : ["attempting fungicide treatment", "keeping infected plants"],
        "urgency_modifier"     : 2.0,
        "recovery_timeline"    : "No recovery — focus on preventing spread to healthy plants",
    },

    "Tomato mosaic virus": {
        "pathogen_type"        : "viral",
        "pathogen_name"        : "Tomato Mosaic Virus (ToMV)",
        "spread_risk"          : "high",
        "environmental_triggers": ["mechanical contact", "contaminated tools", "infected transplants"],
        "primary_actions"      : [
            "Remove and isolate infected plants from the field",
            "Disinfect all tools with 10% bleach solution or 70% ethanol",
            "Wash hands thoroughly before handling healthy plants",
        ],
        "secondary_actions"    : [
            "Source certified virus-free transplants for replanting",
            "Control aphid populations which may serve as secondary vectors",
            "Avoid tobacco use near plants — tobacco can carry ToMV",
        ],
        "monitoring_actions"   : [
            "Inspect all adjacent plants for mosaic patterns daily",
            "Monitor for symptom development on recently handled plants",
        ],
        "contraindications"    : ["continuing to handle infected and healthy plants together"],
        "urgency_modifier"     : 1.5,
        "recovery_timeline"    : "No recovery — prevent further mechanical transmission",
    },

    "healthy": {
        "pathogen_type"        : "none",
        "pathogen_name"        : "N/A",
        "spread_risk"          : "none",
        "environmental_triggers": [],
        "primary_actions"      : [
            "Continue current cultural practices",
            "Maintain regular monitoring schedule",
        ],
        "secondary_actions"    : [
            "Ensure adequate nutrition and balanced fertilisation",
            "Check irrigation uniformity",
        ],
        "monitoring_actions"   : [
            "Inspect weekly as part of routine crop monitoring",
        ],
        "contraindications"    : ["unnecessary pesticide application"],
        "urgency_modifier"     : 0.0,
        "recovery_timeline"    : "N/A — plant is healthy",
    },
}

# Potato diseases use the same pathogen logic as tomato equivalents
KNOWLEDGE_BASE["Early blight (Potato)"] = copy.deepcopy(KNOWLEDGE_BASE["Early blight"])
KNOWLEDGE_BASE["Late blight (Potato)"]  = copy.deepcopy(KNOWLEDGE_BASE["Late blight"])

# Generic fallback for any class not explicitly in the knowledge base
KNOWLEDGE_BASE["__default__"] = {
    "pathogen_type"        : "unknown",
    "pathogen_name"        : "Unknown pathogen",
    "spread_risk"          : "medium",
    "environmental_triggers": ["unknown"],
    "primary_actions"      : [
        "Consult an agricultural extension officer for diagnosis",
        "Isolate affected plants as a precaution",
    ],
    "secondary_actions"    : ["Document symptoms with photographs for expert review"],
    "monitoring_actions"   : ["Observe every 3 days and record any changes"],
    "contraindications"    : [],
    "urgency_modifier"     : 1.0,
    "recovery_timeline"    : "Unknown — expert consultation required",
}

print(f"Knowledge base loaded: {len(KNOWLEDGE_BASE) - 1} disease entries + 1 default")

# Save as JSON for Phase 4 and FastAPI use
with open(f"{CONFIG['plan_output_dir']}/json/knowledge_base.json", "w") as f:
    json.dump(KNOWLEDGE_BASE, f, indent=2)
print("Knowledge base saved to outputs/planning/json/knowledge_base.json")

---
## Cell 8 — Structured State Generator

The state generator converts the raw model output into a symbolic state representation that the planner can reason over. This is the neuro-symbolic bridge: neural network outputs become symbolic predicates.

The state includes computed fields that the model does not directly output, such as `infection_spread` (derived from the Grad-CAM focus score) and `treatment_urgency` (derived from severity, spread risk, and the knowledge base urgency modifier). These computed fields are what allow the planner to produce different action sequences for the same disease at different severities.

In [ ]:
class UrgencyLevel(Enum):
    IMMEDIATE = "immediate"   # act within 24 hours
    HIGH      = "high"        # act within 48 hours
    MEDIUM    = "medium"      # act within 1 week
    LOW       = "low"         # routine monitoring
    NONE      = "none"        # plant is healthy


@dataclass
class DiseaseState:
    """
    Symbolic state representation of a plant disease diagnosis.

    This is the planning engine's input. It is derived from the neural
    network's prediction and the Grad-CAM spatial analysis. Every field
    has a defined type so that the planner can apply rules deterministically.
    """
    # From neural network
    plant            : str
    disease_label    : str          # full label string
    disease_type     : str          # parsed disease name
    pathogen_type    : str          # fungal / bacterial / viral / pest / none
    confidence       : float
    severity         : str          # high / medium / low
    is_healthy       : bool

    # From Grad-CAM analysis
    focus_score      : float        # spatial concentration of activation
    infection_spread : str          # localised / moderate / widespread

    # Derived / computed
    spread_risk      : str
    urgency          : UrgencyLevel
    urgency_score    : float        # continuous score for ranking
    treatment_class  : str          # primary treatment category
    environmental_triggers: List[str]
    knowledge_entry  : Dict

    def to_dict(self):
        d = asdict(self)
        d["urgency"] = self.urgency.value
        d.pop("knowledge_entry")       # not serialisable directly
        return d


def build_state(prediction, heatmap):
    """
    Convert a predict() output and a Grad-CAM heatmap into a DiseaseState.

    This function is the neuro-symbolic bridge: it takes continuous-valued
    neural network outputs and converts them into discrete symbolic predicates
    that the rule-based planner can act on.
    """
    disease_type = prediction["disease_type"]
    confidence   = prediction["confidence"]
    severity     = prediction["severity"]
    is_healthy   = prediction["is_healthy"]

    # Retrieve knowledge base entry (use default if disease not found)
    kb = KNOWLEDGE_BASE.get(disease_type,
         KNOWLEDGE_BASE.get("healthy" if is_healthy else "__default__"))

    # Grad-CAM focus score → infection spread category
    focus = gradcam_engine.focus_score(heatmap)
    if focus < 0.06:
        infection_spread = "localised"
    elif focus < 0.15:
        infection_spread = "moderate"
    else:
        infection_spread = "widespread"

    # Compute urgency score: confidence × severity_weight × urgency_modifier
    severity_weights = {"high": 3.0, "medium": 2.0, "low": 1.0}
    spread_weights   = {"critical": 3.0, "high": 2.0, "medium": 1.5, "low": 1.0, "none": 0.0}
    spread_risk      = kb["spread_risk"]

    urgency_score = (
        confidence *
        severity_weights.get(severity, 1.0) *
        spread_weights.get(spread_risk, 1.0) *
        kb["urgency_modifier"]
    )

    # Map urgency score to discrete level
    if is_healthy:
        urgency = UrgencyLevel.NONE
    elif urgency_score >= 4.5:
        urgency = UrgencyLevel.IMMEDIATE
    elif urgency_score >= 3.0:
        urgency = UrgencyLevel.HIGH
    elif urgency_score >= 1.5:
        urgency = UrgencyLevel.MEDIUM
    else:
        urgency = UrgencyLevel.LOW

    # Determine primary treatment class
    pathogen_type   = kb["pathogen_type"]
    treatment_class = {
        "fungal"    : "fungicide",
        "oomycete"  : "systemic_fungicide",
        "bacterial" : "bactericide",
        "viral"     : "vector_control",
        "pest"      : "miticide_or_insecticide",
        "none"      : "monitoring_only",
        "unknown"   : "expert_consultation",
    }.get(pathogen_type, "expert_consultation")

    return DiseaseState(
        plant                = prediction["plant"],
        disease_label        = prediction["disease"],
        disease_type         = disease_type,
        pathogen_type        = pathogen_type,
        confidence           = confidence,
        severity             = severity,
        is_healthy           = is_healthy,
        focus_score          = round(focus, 4),
        infection_spread     = infection_spread,
        spread_risk          = spread_risk,
        urgency              = urgency,
        urgency_score        = round(urgency_score, 4),
        treatment_class      = treatment_class,
        environmental_triggers = kb["environmental_triggers"],
        knowledge_entry      = kb,
    )


print("DiseaseState and build_state() ready")

---
## Cell 9 — Rule-Based Planning Engine

The planner takes a `DiseaseState` and produces an ordered `TreatmentPlan`. The ordering logic is the scientific contribution here:

1. Containment actions always come before treatment actions. You cannot effectively treat a disease that is still spreading.
2. Primary actions come before secondary actions. Secondary actions reinforce primaries but do not replace them.
3. Monitoring actions are always last. They verify that the plan is working.
4. Urgency gates the plan: immediate urgency adds an isolation step as the very first action regardless of disease type.
5. Confidence gating: if confidence is below 0.70, the plan includes an expert verification step before any chemical treatment.

Each action in the plan is annotated with a rationale string that traces back to the model's evidence — connecting the symbolic plan to the neural network's output.

In [ ]:
@dataclass
class PlanAction:
    """A single action in a treatment plan."""
    step        : int
    action      : str
    category    : str     # containment / primary / secondary / monitoring / verification
    rationale   : str
    priority    : float   # higher = more important
    timeframe   : str     # when this action should be taken

    def to_dict(self):
        return asdict(self)


@dataclass
class TreatmentPlan:
    """Complete treatment plan output by the planning engine."""
    state          : DiseaseState
    actions        : List[PlanAction]
    plan_summary   : str
    urgency_label  : str
    treatment_class: str
    recovery_timeline: str
    contraindications: List[str]
    confidence_note : str

    def to_dict(self):
        return {
            "state"            : self.state.to_dict(),
            "actions"          : [a.to_dict() for a in self.actions],
            "plan_summary"     : self.plan_summary,
            "urgency_label"    : self.urgency_label,
            "treatment_class"  : self.treatment_class,
            "recovery_timeline": self.recovery_timeline,
            "contraindications": self.contraindications,
            "confidence_note"  : self.confidence_note,
        }


class RuleBasedPlanner:
    """
    Explainable rule-based sequential planning engine.

    The planner applies a deterministic rule set to a DiseaseState and
    produces a sequenced, annotated TreatmentPlan. Every action is
    traceable to the model's confidence, severity assessment, and
    Grad-CAM spatial analysis.

    Design principle: the planner is intentionally symbolic and auditable.
    Phase 4 will replace individual rules with PDDL operators while
    keeping the same state/plan schema.
    """

    # Timeframe strings used in action annotations
    TIMEFRAMES = {
        UrgencyLevel.IMMEDIATE : "Within 24 hours",
        UrgencyLevel.HIGH      : "Within 48 hours",
        UrgencyLevel.MEDIUM    : "Within this week",
        UrgencyLevel.LOW       : "Within 2 weeks",
        UrgencyLevel.NONE      : "Routine — no urgency",
    }

    def plan(self, state: DiseaseState) -> TreatmentPlan:
        """Generate a TreatmentPlan from a DiseaseState."""

        actions = []
        step    = 1
        tf      = self.TIMEFRAMES[state.urgency]
        kb      = state.knowledge_entry

        # Rule 0 — Healthy plant: return maintenance plan immediately
        if state.is_healthy:
            for action in kb["primary_actions"] + kb["secondary_actions"]:
                actions.append(PlanAction(
                    step      = step,
                    action    = action,
                    category  = "monitoring",
                    rationale = f"Plant assessed as healthy with {state.confidence:.1%} confidence.",
                    priority  = 1.0,
                    timeframe = "Routine schedule"
                ))
                step += 1
            return self._build_plan(state, actions, kb)

        # Rule 1 — Low confidence: add expert verification before treatment
        if state.confidence < 0.70:
            actions.append(PlanAction(
                step      = step,
                action    = "Obtain expert agronomist or extension officer verification before applying treatments",
                category  = "verification",
                rationale = (
                    f"Model confidence is {state.confidence:.1%}, below the 70% treatment threshold. "
                    f"Applying chemical treatments without high-confidence diagnosis risks unnecessary "
                    f"cost and environmental impact."
                ),
                priority  = 10.0,
                timeframe = tf
            ))
            step += 1

        # Rule 2 — Immediate urgency: isolate before any treatment
        if state.urgency == UrgencyLevel.IMMEDIATE:
            actions.append(PlanAction(
                step      = step,
                action    = "Physically isolate infected plants from the rest of the crop immediately",
                category  = "containment",
                rationale = (
                    f"{state.disease_type} has critical spread risk. "
                    f"The Grad-CAM analysis shows {state.infection_spread} infection pattern. "
                    f"Isolation prevents transmission while treatment is being prepared."
                ),
                priority  = 9.5,
                timeframe = "Immediately"
            ))
            step += 1

        # Rule 3 — Widespread infection: add field-level containment
        if state.infection_spread == "widespread" and not state.is_healthy:
            actions.append(PlanAction(
                step      = step,
                action    = "Apply preventive treatment to all neighbouring plants regardless of visible symptoms",
                category  = "containment",
                rationale = (
                    f"Grad-CAM shows widespread activation ({state.focus_score:.3f} focus score), "
                    f"indicating infection has spread beyond the immediate lesion areas. "
                    f"Preventive treatment of neighbours reduces further spread."
                ),
                priority  = 9.0,
                timeframe = tf
            ))
            step += 1

        # Rule 4 — Viral disease: plant removal is primary action
        if state.pathogen_type == "viral":
            actions.append(PlanAction(
                step      = step,
                action    = "Remove and destroy all infected plants — viral diseases have no chemical cure",
                category  = "primary",
                rationale = (
                    f"{state.disease_type} is caused by {kb['pathogen_name']}, a virus. "
                    f"No registered chemical treatment can eliminate viral infection once established. "
                    f"Plant removal is the only way to prevent spread to healthy crops."
                ),
                priority  = 8.5,
                timeframe = tf
            ))
            step += 1

        # Rule 5 — Primary actions from knowledge base
        for i, action in enumerate(kb["primary_actions"]):
            # Skip plant removal for viral — already added in Rule 4
            if state.pathogen_type == "viral" and "remove" in action.lower():
                continue
            priority = 8.0 - (i * 0.3)  # decreasing priority within primaries
            actions.append(PlanAction(
                step      = step,
                action    = action,
                category  = "primary",
                rationale = (
                    f"Primary treatment for {state.pathogen_type} disease ({state.disease_type}). "
                    f"Confidence: {state.confidence:.1%}. Severity: {state.severity}."
                ),
                priority  = priority,
                timeframe = tf
            ))
            step += 1

        # Rule 6 — Secondary actions (applied at medium+ severity)
        if state.severity in ["high", "medium"]:
            for i, action in enumerate(kb["secondary_actions"]):
                actions.append(PlanAction(
                    step      = step,
                    action    = action,
                    category  = "secondary",
                    rationale = (
                        f"Supportive measure for {state.severity}-severity {state.disease_type}. "
                        f"Addresses environmental conditions that favour disease spread: "
                        f"{', '.join(state.environmental_triggers[:2])}."
                    ),
                    priority  = 5.0 - (i * 0.2),
                    timeframe = tf
                ))
                step += 1

        # Rule 7 — Monitoring actions (always added, timeframe scaled by urgency)
        monitoring_tf = {
            UrgencyLevel.IMMEDIATE : "Every 2 days",
            UrgencyLevel.HIGH      : "Every 3 days",
            UrgencyLevel.MEDIUM    : "Every 5-7 days",
            UrgencyLevel.LOW       : "Weekly",
            UrgencyLevel.NONE      : "Weekly routine",
        }[state.urgency]

        for i, action in enumerate(kb["monitoring_actions"]):
            actions.append(PlanAction(
                step      = step,
                action    = action,
                category  = "monitoring",
                rationale = (
                    f"Post-treatment verification. {kb['recovery_timeline']}. "
                    f"Monitoring frequency increased due to {state.urgency.value} urgency level."
                ),
                priority  = 3.0 - (i * 0.1),
                timeframe = monitoring_tf
            ))
            step += 1

        return self._build_plan(state, actions, kb)

    def _build_plan(self, state, actions, kb):
        """Assemble the final TreatmentPlan from the action list and state."""

        if state.confidence >= 0.85:
            confidence_note = (
                f"High confidence ({state.confidence:.1%}). The visual evidence strongly matches "
                f"the learned pattern for {state.disease_type}. This plan can be acted on directly."
            )
        elif state.confidence >= 0.70:
            confidence_note = (
                f"Moderate confidence ({state.confidence:.1%}). The diagnosis is likely correct "
                f"but a second image from a different leaf would strengthen certainty."
            )
        else:
            confidence_note = (
                f"Low confidence ({state.confidence:.1%}). Expert verification is included as "
                f"Step 1 before any chemical treatment is applied."
            )

        plan_summary = (
            f"{state.disease_type} detected in {state.plant} with {state.confidence:.1%} confidence. "
            f"Pathogen type: {state.pathogen_type}. Severity: {state.severity}. "
            f"Infection spread: {state.infection_spread} (Grad-CAM focus score: {state.focus_score:.3f}). "
            f"Treatment urgency: {state.urgency.value.upper()}. "
            f"{len(actions)} actions planned."
        )

        return TreatmentPlan(
            state             = state,
            actions           = actions,
            plan_summary      = plan_summary,
            urgency_label     = state.urgency.value,
            treatment_class   = state.treatment_class,
            recovery_timeline = kb["recovery_timeline"],
            contraindications = kb["contraindications"],
            confidence_note   = confidence_note,
        )


planner = RuleBasedPlanner()
print("RuleBasedPlanner ready")

---
## Cell 10 — Counterfactual Planner

The counterfactual planner answers: *what would the treatment plan look like under different assumptions?* It generates three alternative plans by modifying the disease state:

1. **Lower severity:** same disease but one severity level down
2. **Lower confidence:** same disease but confidence at the medium threshold boundary
3. **Earlier detection:** same disease but with localised rather than current infection spread

Comparing the baseline plan with its counterfactuals gives the farmer actionable insight: the difference between the localised-detection plan and the current plan shows exactly what was lost by delayed identification.

In [ ]:
class CounterfactualPlanner:
    """
    Generates alternative treatment plans under modified state assumptions.

    Each counterfactual modifies exactly one state variable while holding
    all others constant, then runs the full planner on the modified state.
    This produces interpretable what-if explanations grounded in the plan
    structure rather than in model internals.
    """

    def __init__(self, planner: RuleBasedPlanner):
        self.planner = planner

    def generate(self, baseline_state: DiseaseState) -> Dict:
        """
        Generate counterfactual plans for three modified states.

        Returns a dict with keys 'lower_severity', 'lower_confidence',
        'earlier_detection', each containing a TreatmentPlan.
        """
        counterfactuals = {}

        # Counterfactual 1 — lower severity
        severity_ladder = {"high": "medium", "medium": "low", "low": "low"}
        lower_sev       = copy.deepcopy(baseline_state)
        lower_sev.severity     = severity_ladder[baseline_state.severity]
        lower_sev.urgency_score = baseline_state.urgency_score * 0.6
        lower_sev.urgency = self._reclassify_urgency(lower_sev)
        counterfactuals["lower_severity"] = {
            "description" : f"If the infection were {lower_sev.severity} severity instead of {baseline_state.severity}",
            "plan"        : self.planner.plan(lower_sev)
        }

        # Counterfactual 2 — lower confidence (boundary case)
        lower_conf              = copy.deepcopy(baseline_state)
        lower_conf.confidence   = 0.65
        lower_conf.urgency_score = baseline_state.urgency_score * 0.65 / baseline_state.confidence
        lower_conf.urgency = self._reclassify_urgency(lower_conf)
        counterfactuals["lower_confidence"] = {
            "description" : "If the model confidence were 65% (below treatment threshold)",
            "plan"        : self.planner.plan(lower_conf)
        }

        # Counterfactual 3 — earlier detection (localised spread)
        earlier               = copy.deepcopy(baseline_state)
        earlier.infection_spread = "localised"
        earlier.focus_score      = 0.04
        earlier.severity         = "low" if baseline_state.severity == "high" else baseline_state.severity
        earlier.urgency_score    = baseline_state.urgency_score * 0.4
        earlier.urgency = self._reclassify_urgency(earlier)
        counterfactuals["earlier_detection"] = {
            "description" : "If the disease had been detected at an earlier, localised stage",
            "plan"        : self.planner.plan(earlier)
        }

        return counterfactuals

    def _reclassify_urgency(self, state):
        if state.is_healthy:
            return UrgencyLevel.NONE
        if state.urgency_score >= 4.5:
            return UrgencyLevel.IMMEDIATE
        if state.urgency_score >= 3.0:
            return UrgencyLevel.HIGH
        if state.urgency_score >= 1.5:
            return UrgencyLevel.MEDIUM
        return UrgencyLevel.LOW


cf_planner = CounterfactualPlanner(planner)
print("CounterfactualPlanner ready")

---
## Cell 11 — Human-Aware Explanation Adapter

The same treatment plan is presented differently depending on who is reading it. A farmer needs simple, action-first language with clear timeframes. A researcher needs the full reasoning chain with confidence values and method attribution. An agronomist needs technical terminology and dosage guidance framing.

This is a lightweight version of the user-adaptive explainability that Phase 5 will implement more fully using a user model.

In [ ]:
class UserMode(Enum):
    FARMER     = "farmer"
    AGRONOMIST = "agronomist"
    RESEARCHER = "researcher"


class HumanAwareAdapter:
    """
    Adapts TreatmentPlan language and detail level to the user's expertise.

    Three modes are supported:
      FARMER     — simple language, action-first, no technical jargon
      AGRONOMIST — technical terminology, dosage framing, full action list
      RESEARCHER — includes confidence values, method attribution, all metadata
    """

    def render(self, plan: TreatmentPlan, mode: UserMode) -> str:
        if mode == UserMode.FARMER:
            return self._render_farmer(plan)
        elif mode == UserMode.AGRONOMIST:
            return self._render_agronomist(plan)
        elif mode == UserMode.RESEARCHER:
            return self._render_researcher(plan)

    def _render_farmer(self, plan: TreatmentPlan) -> str:
        s   = plan.state
        out = []

        out.append("PLANT HEALTH REPORT")
        out.append("=" * 40)

        if s.is_healthy:
            out.append(f"Good news: your {s.plant} plant looks healthy.")
            out.append("Keep doing what you are doing.")
        else:
            urgency_words = {
                UrgencyLevel.IMMEDIATE : "ACT TODAY — this is serious.",
                UrgencyLevel.HIGH      : "Act within the next 2 days.",
                UrgencyLevel.MEDIUM    : "Act this week.",
                UrgencyLevel.LOW       : "No rush — act when convenient.",
            }
            out.append(f"Problem found: {s.disease_type}")
            out.append(f"Your {s.plant} plant has a disease.")
            out.append(f"How serious: {s.severity.upper()}")
            out.append(f"{urgency_words.get(s.urgency, '')}")
            out.append("")
            out.append("What to do:")
            for action in plan.actions:
                if action.category in ["primary", "containment", "verification"]:
                    out.append(f"  Step {action.step}: {action.action}")
                    out.append(f"  When: {action.timeframe}")
                    out.append("")

            if plan.contraindications:
                out.append("Do NOT do these:")
                for c in plan.contraindications:
                    out.append(f"  - {c}")

        return "\n".join(out)

    def _render_agronomist(self, plan: TreatmentPlan) -> str:
        s   = plan.state
        out = []

        out.append("AGRONOMIC TREATMENT PLAN")
        out.append("=" * 50)
        out.append(f"Crop        : {s.plant}")
        out.append(f"Diagnosis   : {s.disease_type} ({s.pathogen_type})")
        out.append(f"Pathogen    : {s.knowledge_entry['pathogen_name']}")
        out.append(f"Severity    : {s.severity}")
        out.append(f"Spread risk : {s.spread_risk}")
        out.append(f"Urgency     : {s.urgency.value.upper()}")
        out.append(f"Treatment   : {s.treatment_class.replace('_', ' ')}")
        out.append(f"Recovery    : {plan.recovery_timeline}")
        out.append("")

        categories = ["containment", "verification", "primary", "secondary", "monitoring"]
        for cat in categories:
            cat_actions = [a for a in plan.actions if a.category == cat]
            if cat_actions:
                out.append(f"{cat.upper()} ACTIONS")
                out.append("-" * 30)
                for a in cat_actions:
                    out.append(f"  {a.step}. {a.action}")
                    out.append(f"     Timeframe: {a.timeframe}")
                out.append("")

        if plan.contraindications:
            out.append("CONTRAINDICATIONS")
            out.append("-" * 30)
            for c in plan.contraindications:
                out.append(f"  - {c}")
            out.append("")

        out.append(f"Environmental triggers: {', '.join(s.environmental_triggers)}")

        return "\n".join(out)

    def _render_researcher(self, plan: TreatmentPlan) -> str:
        s   = plan.state
        out = []

        out.append("EXPLAINABLE PLANNING OUTPUT — ExplainPlan-Vision Phase 3")
        out.append("=" * 60)
        out.append("VISION MODULE OUTPUT")
        out.append(f"  Model          : EfficientNet-B0 (Phase 1)")
        out.append(f"  Predicted class: {s.disease_label}")
        out.append(f"  Confidence     : {s.confidence:.4f}")
        out.append(f"  Severity       : {s.severity} (threshold: high>=0.85, medium>=0.60)")
        out.append("")
        out.append("XAI MODULE OUTPUT")
        out.append(f"  Grad-CAM focus score   : {s.focus_score:.4f}")
        out.append(f"  Infection spread       : {s.infection_spread}")
        out.append("")
        out.append("SYMBOLIC STATE")
        out.append(f"  Pathogen type    : {s.pathogen_type}")
        out.append(f"  Spread risk      : {s.spread_risk}")
        out.append(f"  Urgency score    : {s.urgency_score:.4f}")
        out.append(f"  Urgency level    : {s.urgency.value}")
        out.append(f"  Treatment class  : {s.treatment_class}")
        out.append("")
        out.append("PLANNING ENGINE OUTPUT")
        out.append(f"  Total actions    : {len(plan.actions)}")
        out.append(f"  Plan summary     : {plan.plan_summary}")
        out.append("")
        out.append("ACTION SEQUENCE WITH RATIONALE")
        for a in plan.actions:
            out.append(f"  [{a.step}] [{a.category.upper()}] {a.action}")
            out.append(f"      Rationale : {a.rationale}")
            out.append(f"      Priority  : {a.priority:.1f}")
            out.append(f"      Timeframe : {a.timeframe}")
            out.append("")

        out.append("CONFIDENCE ASSESSMENT")
        out.append(f"  {plan.confidence_note}")
        out.append("")
        out.append("CONTRAINDICATIONS")
        for c in plan.contraindications:
            out.append(f"  - {c}")

        return "\n".join(out)


adapter = HumanAwareAdapter()
print("HumanAwareAdapter ready (FARMER / AGRONOMIST / RESEARCHER modes)")

---
## Cell 12 — Full end-to-end pipeline function

This is the primary interface that the Phase 5 human-aware module and the FastAPI deployment will call. It takes a raw image path and returns the complete enriched output: prediction, visual explanation, symbolic state, treatment plan, counterfactuals, and all three user-mode renderings.

In [ ]:
def run_pipeline(image_path: str, verbose: bool = False) -> Dict:
    """
    Full ExplainPlan-Vision pipeline: image → treatment plan.

    Parameters
    ----------
    image_path : str — path to the input leaf image
    verbose    : bool — print intermediate outputs

    Returns
    -------
    A dictionary containing:
      prediction      — Phase 1 structured output
      heatmap         — Grad-CAM numpy array (224, 224)
      gradcam_overlay — Grad-CAM overlay image (224, 224, 3)
      state           — DiseaseState (symbolic representation)
      plan            — TreatmentPlan
      counterfactuals — dict of three alternative plans
      explanations    — dict with FARMER/AGRONOMIST/RESEARCHER text
      image_rgb       — original image
    """
    image_rgb  = load_image_rgb(image_path)
    prediction = predict(image_rgb)

    heatmap, gc_overlay = gradcam_engine.generate(image_rgb)

    state = build_state(prediction, heatmap)

    plan  = planner.plan(state)

    counterfactuals = cf_planner.generate(state)

    explanations = {
        "farmer"     : adapter.render(plan, UserMode.FARMER),
        "agronomist" : adapter.render(plan, UserMode.AGRONOMIST),
        "researcher" : adapter.render(plan, UserMode.RESEARCHER),
    }

    if verbose:
        print(explanations["researcher"])

    return {
        "prediction"      : prediction,
        "heatmap"         : heatmap,
        "gradcam_overlay" : gc_overlay,
        "state"           : state,
        "plan"            : plan,
        "counterfactuals" : counterfactuals,
        "explanations"    : explanations,
        "image_rgb"       : image_rgb,
    }


print("run_pipeline() ready")
print()
print("Output schema for Phase 5 and FastAPI:")
print("  run_pipeline(image_path) -> {")
print("    prediction, heatmap, gradcam_overlay,")
print("    state (DiseaseState), plan (TreatmentPlan),")
print("    counterfactuals {lower_severity, lower_confidence, earlier_detection},")
print("    explanations {farmer, agronomist, researcher}")
print("  }")

---
## Cell 13 — End-to-end demo on representative images

We run the full pipeline on one image per disease class and display the structured outputs. This cell generates the primary demo figure for the research paper and the GitHub README.

In [ ]:
dataset_path = Path(DATASET_PATH)
random.seed(CONFIG["seed"])

# Select one representative image per class
demo_samples = []
for cls in classes:
    cls_dir = dataset_path / cls
    imgs    = [f for f in cls_dir.iterdir() if f.suffix.lower() in [".jpg",".jpeg",".png"]]
    if imgs:
        demo_samples.append({"path": str(random.choice(imgs)), "label": cls})

print(f"Running full pipeline on {len(demo_samples)} images...")
pipeline_results = []

for sample in tqdm(demo_samples, desc="Pipeline"):
    result = run_pipeline(sample["path"])
    pipeline_results.append(result)

print(f"Pipeline complete for {len(pipeline_results)} images")

---
## Cell 14 — Plan visualisation figure

For each demo image we display the original leaf, the Grad-CAM overlay, and the treatment plan as a structured card. This is the key figure that shows the system going from image to actionable plan in one unified view.

In [ ]:
def render_plan_card(ax, plan: TreatmentPlan, mode="compact"):
    """Render a treatment plan as text inside a matplotlib axes."""
    s    = plan.state
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    urgency_colors = {
        "immediate": "#d32f2f",
        "high"     : "#f57c00",
        "medium"   : "#fbc02d",
        "low"      : "#388e3c",
        "none"     : "#1976d2",
    }
    color = urgency_colors.get(s.urgency.value, "gray")

    # Header
    ax.text(0.5, 0.97, s.disease_type, ha="center", va="top",
            fontsize=7, fontweight="bold", transform=ax.transAxes)
    ax.text(0.5, 0.91, f"Urgency: {s.urgency.value.upper()}",
            ha="center", va="top", fontsize=6.5, color=color,
            fontweight="bold", transform=ax.transAxes)
    ax.text(0.5, 0.85, f"Conf: {s.confidence:.2f}  Sev: {s.severity}  Spread: {s.infection_spread}",
            ha="center", va="top", fontsize=5.5, color="#444", transform=ax.transAxes)

    # Actions (show only primary + containment for compact view)
    y = 0.76
    primary = [a for a in plan.actions if a.category in ["primary", "containment", "verification"]]
    for a in primary[:4]:  # max 4 actions in compact view
        text = textwrap.fill(f"{a.step}. {a.action}", width=38)
        ax.text(0.03, y, text, ha="left", va="top", fontsize=4.8,
                transform=ax.transAxes, color="#222")
        y -= 0.15 * (text.count("\n") + 1)
        if y < 0.15:
            break

    # Recovery timeline at bottom
    ax.text(0.5, 0.04, plan.recovery_timeline[:50],
            ha="center", va="bottom", fontsize=4.5, color="#666",
            style="italic", transform=ax.transAxes)

    # Border
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(color)
        spine.set_linewidth(1.5)


# --- Main figure ---
n_show = min(8, len(pipeline_results))
show   = pipeline_results[:n_show]

fig, axes = plt.subplots(n_show, 3, figsize=(14, n_show * 3.0))

axes[0, 0].set_title("Original Image",    fontsize=9, fontweight="bold")
axes[0, 1].set_title("Grad-CAM++ (XAI)",  fontsize=9, fontweight="bold")
axes[0, 2].set_title("Treatment Plan",    fontsize=9, fontweight="bold")

for i, res in enumerate(show):
    original = cv2.resize(res["image_rgb"], (224, 224))
    axes[i, 0].imshow(original)
    axes[i, 0].set_ylabel(
        f"{res['state'].plant}\n{res['state'].disease_type[:18]}",
        fontsize=7, rotation=0, labelpad=65, va="center"
    )
    axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])

    axes[i, 1].imshow(res["gradcam_overlay"])
    axes[i, 1].set_xlabel(
        f"focus={res['state'].focus_score:.3f}  spread={res['state'].infection_spread}",
        fontsize=6.5
    )
    axes[i, 1].set_xticks([]); axes[i, 1].set_yticks([])

    render_plan_card(axes[i, 2], res["plan"])

plt.suptitle(
    "ExplainPlan-Vision Phase 3 — Full Pipeline: Image → Explanation → Plan",
    fontsize=12, fontweight="bold", y=1.005
)
plt.tight_layout()
plt.savefig(f"{CONFIG['plan_output_dir']}/figures/pipeline_demo.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Pipeline demo figure saved")

---
## Cell 15 — Counterfactual plan comparison figure

This figure shows the baseline plan alongside its three counterfactuals for a single high-urgency disease case. It visualises the key research claim: the planning engine's outputs are sensitive to the state variables in a principled, auditable way.

In [ ]:
# Select the result with highest urgency score for the counterfactual demo
cf_demo = max(pipeline_results, key=lambda r: r["state"].urgency_score)
cf_dict = cf_demo["counterfactuals"]

fig, axes = plt.subplots(1, 4, figsize=(18, 6))

scenarios = [
    ("Baseline",          cf_demo["plan"],                              cf_demo["state"]),
    ("Lower Severity",    cf_dict["lower_severity"]["plan"],            cf_dict["lower_severity"]["plan"].state),
    ("Lower Confidence",  cf_dict["lower_confidence"]["plan"],          cf_dict["lower_confidence"]["plan"].state),
    ("Earlier Detection", cf_dict["earlier_detection"]["plan"],         cf_dict["earlier_detection"]["plan"].state),
]

for ax, (title, plan, state) in zip(axes, scenarios):
    ax.set_title(
        f"{title}\nUrgency: {state.urgency.value.upper()}  |  Actions: {len(plan.actions)}",
        fontsize=9, fontweight="bold"
    )
    render_plan_card(ax, plan)

plt.suptitle(
    f"Counterfactual Plan Comparison — {cf_demo['state'].disease_type} in {cf_demo['state'].plant}",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.savefig(f"{CONFIG['plan_output_dir']}/figures/counterfactual_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()
print(f"Counterfactual demo: {cf_demo['state'].disease_type}")
print(f"Baseline actions  : {len(cf_demo['plan'].actions)}")
for key, cf in cf_dict.items():
    print(f"{key:20s}: {len(cf['plan'].actions)} actions  urgency={cf['plan'].state.urgency.value}")

---
## Cell 16 — Human-aware explanation demo

Print all three user-mode renderings for the same plan to show the adaptation in action.

In [ ]:
demo_result = cf_demo  # use the same high-urgency case

for mode_name, text in demo_result["explanations"].items():
    print("#" * 60)
    print(f"USER MODE: {mode_name.upper()}")
    print("#" * 60)
    print(text)
    print()

---
## Cell 17 — Plan statistics across all classes

This table gives an overview of how the planner behaves across all 15 disease classes. It is the basis for the planning evaluation section of the research paper.

In [ ]:
rows = []
for res in pipeline_results:
    s = res["state"]
    p = res["plan"]
    rows.append({
        "plant"            : s.plant,
        "disease"          : s.disease_type,
        "pathogen_type"    : s.pathogen_type,
        "confidence"       : s.confidence,
        "severity"         : s.severity,
        "infection_spread" : s.infection_spread,
        "urgency_score"    : s.urgency_score,
        "urgency_level"    : s.urgency.value,
        "total_actions"    : len(p.actions),
        "primary_actions"  : sum(1 for a in p.actions if a.category == "primary"),
        "containment"      : sum(1 for a in p.actions if a.category == "containment"),
        "monitoring"       : sum(1 for a in p.actions if a.category == "monitoring"),
        "treatment_class"  : s.treatment_class,
    })

stats_df = pd.DataFrame(rows)

print("PLANNING ENGINE STATISTICS ACROSS ALL CLASSES")
print("=" * 80)
print(stats_df[["disease", "confidence", "severity", "urgency_level",
                "urgency_score", "total_actions", "treatment_class"]].to_string(index=False))
print()
print(f"Mean confidence    : {stats_df['confidence'].mean():.4f}")
print(f"Mean actions/plan  : {stats_df['total_actions'].mean():.1f}")
print(f"Urgency breakdown  :")
print(stats_df["urgency_level"].value_counts().to_string())

stats_df.to_csv(f"{CONFIG['plan_output_dir']}/reports/plan_statistics.csv", index=False)
print("\nStatistics saved to outputs/planning/reports/plan_statistics.csv")

---
## Cell 18 — Urgency distribution figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1 — Urgency score by disease
sorted_df = stats_df.sort_values("urgency_score", ascending=True)
urgency_palette = {
    "immediate": "#d32f2f", "high": "#f57c00",
    "medium": "#fbc02d",    "low": "#388e3c", "none": "#1976d2"
}
colors = [urgency_palette.get(u, "gray") for u in sorted_df["urgency_level"]]
axes[0].barh(range(len(sorted_df)), sorted_df["urgency_score"], color=colors)
axes[0].set_yticks(range(len(sorted_df)))
axes[0].set_yticklabels(
    [d[:22] for d in sorted_df["disease"]], fontsize=7
)
axes[0].set_xlabel("Urgency Score")
axes[0].set_title("Urgency Score by Disease", fontweight="bold")
legend_patches = [mpatches.Patch(color=c, label=l) for l, c in urgency_palette.items()]
axes[0].legend(handles=legend_patches, fontsize=7, loc="lower right")

# Plot 2 — Action count by category
categories = ["containment", "primary_actions", "monitoring"]
cat_means  = [stats_df[c].mean() for c in categories]
axes[1].bar(categories, cat_means, color=["#d32f2f", "#1976d2", "#388e3c"])
axes[1].set_title("Mean Actions per Category", fontweight="bold")
axes[1].set_ylabel("Mean action count")

# Plot 3 — Pathogen type breakdown
pathogen_counts = stats_df["pathogen_type"].value_counts()
axes[2].pie(
    pathogen_counts.values,
    labels=pathogen_counts.index,
    autopct="%1.0f%%",
    colors=["#1976d2", "#f57c00", "#d32f2f", "#388e3c", "#9c27b0"]
)
axes[2].set_title("Pathogen Type Distribution", fontweight="bold")

plt.suptitle(
    "Planning Engine Analytics — ExplainPlan-Vision Phase 3",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(f"{CONFIG['plan_output_dir']}/figures/planning_analytics.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Analytics figure saved")

---
## Cell 19 — Export all plans as structured JSON

Every plan is saved as a JSON file. These files are what the FastAPI deployment will return as API responses, and what Phase 5 will read to generate user-adaptive text.

In [ ]:
for res in pipeline_results:
    plan  = res["plan"]
    state = res["state"]

    label_safe = state.disease_label.replace("/", "_")

    # Full plan JSON
    full_output = {
        "plan"           : plan.to_dict(),
        "counterfactuals": {
            k: {
                "description": v["description"],
                "plan"       : v["plan"].to_dict()
            }
            for k, v in res["counterfactuals"].items()
        },
        "explanations": res["explanations"]
    }

    out_path = f"{CONFIG['plan_output_dir']}/json/{label_safe}_plan.json"
    with open(out_path, "w") as f:
        json.dump(full_output, f, indent=2)

print(f"Saved {len(pipeline_results)} plan JSON files")

# Save the API schema
api_schema = {
    "description": "Output schema of run_pipeline() — Phase 3 interface contract",
    "endpoint"   : "POST /api/v1/analyse",
    "input"      : {"image": "multipart file upload"},
    "output": {
        "plan": {
            "state"            : "DiseaseState fields",
            "actions"          : "list of PlanAction",
            "urgency_label"    : "immediate | high | medium | low | none",
            "treatment_class"  : "primary treatment category",
            "recovery_timeline": "expected recovery duration string",
            "contraindications": "list of things not to do",
            "confidence_note"  : "model confidence interpretation",
        },
        "counterfactuals": {
            "lower_severity"    : "plan under reduced severity",
            "lower_confidence"  : "plan under 65% confidence",
            "earlier_detection" : "plan under localised spread",
        },
        "explanations": {
            "farmer"     : "simple language, action-first",
            "agronomist" : "technical, full action list",
            "researcher" : "full metadata, rationale, confidence",
        }
    }
}

with open(f"{CONFIG['plan_output_dir']}/json/api_schema.json", "w") as f:
    json.dump(api_schema, f, indent=2)

print("API schema saved")

---
## Cell 20 — Output directory listing

In [ ]:
for root, dirs, files in os.walk("outputs/planning"):
    level   = root.replace("outputs/planning", "").count(os.sep)
    indent  = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for fname in files:
        size = os.path.getsize(os.path.join(root, fname))
        print(f"  {indent}{fname}  ({size:,} bytes)")

---
## Cell 21 — Phase 3 summary and bridge to Phase 4 and FastAPI

### What Phase 3 delivers

| Component | Implementation | Research contribution |
|-----------|---------------|---------------------|
| Structured State Generator | `build_state()` | Neuro-symbolic bridge — converts neural outputs to symbolic predicates |
| Disease Knowledge Base | `KNOWLEDGE_BASE` JSON | Symbolic domain knowledge encoding agronomic rules |
| Rule-Based Planner | `RuleBasedPlanner` | Deterministic, auditable sequential planning |
| Counterfactual Planner | `CounterfactualPlanner` | What-if reasoning grounded in state variables |
| Human-Aware Adapter | `HumanAwareAdapter` | User-adaptive explanation across three expertise levels |
| Full Pipeline | `run_pipeline()` | End-to-end image → plan interface for deployment |

### Bridge to the next phases

**FastAPI deployment (after Phase 3):** The `run_pipeline()` function is already structured to be a FastAPI endpoint. The API schema saved to `api_schema.json` documents the exact request and response format.

**Phase 4 — Neuro-Symbolic Reasoning:** The `DiseaseState` dataclass maps directly to PDDL predicates. Each field is a typed symbolic variable. Phase 4 will replace the `RuleBasedPlanner` with a PDDL-lite solver that searches over the state space, while keeping the same `DiseaseState` and `TreatmentPlan` schema.

**Phase 5 — Human-Aware Explanations:** The `HumanAwareAdapter` will be replaced by a user-model-driven selector that infers the user's expertise level from their interaction history and selects the appropriate rendering automatically.

---

**GitHub note:** Commit this notebook as `notebooks/phase3_explainable_planning.ipynb`. Save the `outputs/planning/` directory as a Kaggle dataset named `explainplan-phase3-outputs` for use by Phase 4 and the FastAPI deployment notebook.